# Prática 1 - Cliente IoT para o Serviço de Monitoramento

**Curso de IA, Otimização e IoT para Indústrias e Negócios - Do Zero ao MLOps**

No primeiro notebook, executamos o YOLO diretamente no ambiente de experimentação. Agora vamos assumir o papel de um **cliente**: uma aplicação que envia imagens para o serviço e utiliza sua resposta.

Neste exemplo, o cliente simula uma câmera IoT. O modelo não será carregado neste notebook. Ele permanece no servidor FastAPI.

## Arquitetura cliente-servidor

```text
CLIENTE                                     SERVIDOR
este notebook                              servico_patio.py

seleciona uma imagem                        mantém o YOLO carregado
        |                                           ^
        |-------- POST /predict + imagem ---------->|
        |                                           |
        |<----------- resposta JSON ----------------|
        v
exibe contagens, status e caixas
```

O cliente precisa conhecer o endereço e o contrato da API, mas não precisa importar o YOLO nem conhecer a implementação interna do servidor.

## 1. Inicie o servidor

Antes de executar as próximas células, abra um terminal na pasta da prática e mantenha o servidor em execução:

```bash
source .venv/bin/activate
uvicorn servico_patio:app --reload
```

O servidor ficará disponível em `http://localhost:8000`.

> Abrir `/predict` diretamente no navegador não executa uma previsão. A barra de endereços realiza uma requisição `GET`, enquanto `/predict` espera uma requisição `POST` contendo uma imagem. Para testes manuais, use `http://localhost:8000/docs`.

## 2. Bibliotecas do cliente

O cliente usa `requests` para conversar com a API. Observe que não importamos `ultralytics` nem carregamos o modelo YOLO.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from matplotlib.patches import Rectangle
from PIL import Image

## 3. Endereço do serviço

Vamos separar o endereço-base do endpoint. Isso facilita trocar o servidor local por outro endereço no futuro.

In [ ]:
URL_BASE = "http://localhost:8000"
URL_HEALTH = f"{URL_BASE}/health"
URL_PREDICT = f"{URL_BASE}/predict"

print(f"Servidor: {URL_BASE}")
print(f"Endpoint de inferência: {URL_PREDICT}")

## 4. Verificação de saúde

Antes de enviar uma imagem, o cliente pode verificar se o serviço está disponível. Essa requisição utiliza `GET` porque apenas consulta o estado do servidor.

In [ ]:
try:
    resposta_health = requests.get(URL_HEALTH, timeout=10)
    resposta_health.raise_for_status()
    print("Serviço disponível:", resposta_health.json())
except requests.RequestException as erro:
    raise RuntimeError(
        "Não foi possível acessar a API. Confirme se o Uvicorn está em execução."
    ) from erro

## 5. Seleção da imagem

Usaremos as mesmas imagens do experimento para comparar a execução direta com a resposta do serviço.

In [ ]:
caminho_pratica_no_repositorio = Path("M4/mlops/pratica-01-monitoramento-inteligente-de-patio")
raizes_candidatas = [Path.cwd(), *Path.cwd().parents]
pastas_candidatas = []
for raiz in raizes_candidatas:
    pastas_candidatas.extend([
        raiz / "images/patio",
        raiz / caminho_pratica_no_repositorio / "images/patio",
    ])
pasta_imagens = next((pasta for pasta in pastas_candidatas if pasta.exists()), None)

if pasta_imagens is None:
    raise FileNotFoundError(
        "A pasta images/patio não foi encontrada. Abra o Jupyter na raiz do repositório ou na pasta da prática."
    )

caminhos_imagens = [
    pasta_imagens / "patio_estacionamento.jpg",
    pasta_imagens / "patio_pessoas_veiculos.jpg",
    pasta_imagens / "patio_trafego_misto.jpg",
]

for indice, caminho in enumerate(caminhos_imagens):
    print(f"{indice}: {caminho.name}")

In [ ]:
indice_imagem = 0
caminho_imagem = caminhos_imagens[indice_imagem]

imagem_original = Image.open(caminho_imagem).convert("RGB")
plt.figure(figsize=(12, 8))
plt.imshow(imagem_original)
plt.title(f"Imagem que será enviada: {caminho_imagem.name}")
plt.axis("off")
plt.show()

## 6. Envio da imagem com POST

A requisição contém dois tipos de entrada:

- `confianca_minima`, enviado como parâmetro da URL;
- `arquivo`, enviado no corpo da requisição como arquivo multipart.

O nome `arquivo` deve ser exatamente o mesmo definido no endpoint FastAPI.

In [ ]:
def enviar_imagem(caminho, confianca_minima=0.35):
    parametros = {"confianca_minima": confianca_minima}

    with Path(caminho).open("rb") as arquivo_imagem:
        arquivos = {
            "arquivo": (Path(caminho).name, arquivo_imagem, "image/jpeg")
        }
        resposta = requests.post(
            URL_PREDICT,
            params=parametros,
            files=arquivos,
            timeout=120,
        )

    resposta.raise_for_status()
    return resposta.json()

In [ ]:
resultado = enviar_imagem(caminho_imagem, confianca_minima=0.35)
resultado

## 7. Uso da resposta pelo cliente

O servidor respondeu em JSON. Agora o cliente decide como usar esses dados. Primeiro, mostraremos um resumo operacional.

In [ ]:
print(f"Arquivo: {resultado['arquivo']}")
print(f"Modelo no servidor: {resultado['modelo']}")
print(f"Confiança mínima: {resultado['confianca_minima']:.0%}")
print(f"Total de objetos de interesse: {resultado['total']}")
print(f"Contagens: {resultado['contagens']}")
print(f"Status: {resultado['status']}")

In [ ]:
tabela_deteccoes = pd.DataFrame(resultado["deteccoes"])
tabela_deteccoes

## 8. Reconstrução das caixas no cliente

O servidor não devolveu uma imagem pronta. Ele devolveu coordenadas. Isso permite que cada cliente escolha como apresentar ou utilizar as detecções.

Vamos desenhar as caixas recebidas sobre a imagem original.

In [ ]:
def exibir_deteccoes(caminho, deteccoes):
    imagem = Image.open(caminho).convert("RGB")
    figura, eixo = plt.subplots(figsize=(14, 9))
    eixo.imshow(imagem)

    for deteccao in deteccoes:
        x1, y1, x2, y2 = deteccao["caixa_xyxy"]
        largura = x2 - x1
        altura = y2 - y1
        rotulo = f"{deteccao['classe']} {deteccao['confianca']:.0%}"

        caixa = Rectangle(
            (x1, y1),
            largura,
            altura,
            linewidth=2,
            edgecolor="#00ff88",
            facecolor="none",
        )
        eixo.add_patch(caixa)
        eixo.text(
            x1,
            max(0, y1 - 5),
            rotulo,
            color="black",
            fontsize=9,
            bbox={"facecolor": "#00ff88", "alpha": 0.85, "pad": 2},
        )

    eixo.set_title("Detecções recebidas da API")
    eixo.axis("off")
    plt.show()

exibir_deteccoes(caminho_imagem, resultado["deteccoes"] )

## 9. Simulação de envios sucessivos

Uma câmera produziria várias imagens ao longo do tempo. Vamos simular três capturas sequenciais. O modelo continua carregado no servidor enquanto o cliente envia novos arquivos.

In [ ]:
resumos = []

for caminho in caminhos_imagens:
    item = enviar_imagem(caminho, confianca_minima=0.35)
    resumos.append({
        "arquivo": item["arquivo"],
        "total": item["total"],
        "status": item["status"],
        **item["contagens"],
    })

pd.DataFrame(resumos).fillna(0)

## 10. Tratamento de indisponibilidade

Clientes reais precisam lidar com falhas de comunicação. O exemplo abaixo apresenta uma forma simples de capturar erros.

In [ ]:
def enviar_com_tratamento(caminho, confianca_minima=0.35):
    try:
        return enviar_imagem(caminho, confianca_minima)
    except requests.Timeout:
        print("A API demorou mais que o limite esperado.")
    except requests.ConnectionError:
        print("Não foi possível conectar ao servidor.")
    except requests.HTTPError as erro:
        print(f"A API rejeitou a requisição: {erro}")
    return None

## Para praticar

1. Envie as três imagens e compare as respostas.
2. Modifique `confianca_minima` para `0.20`, `0.50` e `0.80`.
3. Observe que o notebook cliente não importa o YOLO. Explique onde o modelo está executando.
4. Interrompa o servidor e execute novamente a verificação de saúde. O que acontece?
5. Explique por que devolver coordenadas pode ser mais flexível que devolver apenas uma imagem anotada.

## Conclusão

Nesta etapa, separamos claramente as responsabilidades:

- o **cliente** escolhe a imagem, envia a requisição e utiliza a resposta;
- o **servidor** mantém o modelo carregado, valida a entrada e executa a inferência;
- a **API** define o contrato de comunicação entre os dois.

Essa separação permite que diferentes aplicações, câmeras ou sistemas utilizem o mesmo modelo sem incorporar sua implementação.